# 🧪 AiStock 性能基准测试

## 测试目标
- ✅ 各模块性能基准
- ✅ 缓存性能对比
- ✅ 数据库查询性能
- ✅ 系统吞吐量测试

## 测试指标
- 响应时间
- 吞吐量
- 内存使用
- CPU 使用

In [ ]:
# 环境设置
import sys
from pathlib import Path
import time
import tracemalloc
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成")

In [ ]:
# 导入测试模块
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from dynamic_price_system.data.data_loader import DataLoader
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

print("✅ 测试模块导入成功")

## 1️⃣ 缓存性能测试

In [ ]:
print("\n【测试 1】缓存性能基准...")

# 测试不同大小的缓存
cache_sizes = [100, 500, 1000, 2000]
results = []

for size in cache_sizes:
    cache = CacheService(max_size=size, ttl=3600)
    
    # 写入测试
    start = time.time()
    for i in range(size):
        cache.set(f'key_{i}', {'value': i})
    write_time = time.time() - start
    
    # 读取测试
    start = time.time()
    for i in range(size):
        cache.get(f'key_{i}')
    read_time = time.time() - start
    
    results.append({
        'size': size,
        'write_time': write_time,
        'read_time': read_time,
        'write_per_sec': size / write_time,
        'read_per_sec': size / read_time
    })

# 显示结果
df = pd.DataFrame(results)
print("\n📊 缓存性能：")
print(df.to_string(index=False))

## 2️⃣ 数据加载性能测试

In [ ]:
print("\n【测试 2】数据加载性能...")

config = ConfigService(system_name='dynamic_price')
cache = CacheService(max_size=1000, ttl=3600)
db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)

data_loader = DataLoader(
    config_service=config,
    cache_service=cache,
    db_main=db,
    enable_cache=True
)

# 测试股票数据加载
test_codes = ['600938', '601899', '300750', '600406', '000400']

print("\n📊 股票数据加载时间：")
for code in test_codes:
    # 首次加载（未命中）
    start = time.time()
    data_loader.load_stock_daily(code, min_days=100)
    time1 = time.time() - start
    
    # 二次加载（命中）
    start = time.time()
    data_loader.load_stock_daily(code, min_days=100)
    time2 = time.time() - start
    
    speedup = time1 / time2 if time2 > 0 else 0
    print(f"  • {code}: 首次{time1:.3f}s | 二次{time2:.3f}s | 提升{speedup:.1f}x")

## 3️⃣ 计算引擎性能测试

In [ ]:
print("\n【测试 3】计算引擎性能...")

import numpy as np

# 生成不同规模的数据
data_sizes = [50, 100, 200, 500]
engine = DynamicPriceEngine(config_service=config, cache_service=cache)

print("\n📊 计算引擎性能：")
for size in data_sizes:
    # 生成模拟数据
    dates = pd.date_range('2025-01-01', periods=size, freq='D')
    df = pd.DataFrame({
        'datetime': dates,
        'open': 40 + np.random.randn(size),
        'high': 40 + np.random.randn(size),
        'low': 40 + np.random.randn(size),
        'close': 40 + np.random.randn(size),
        'volume': np.random.randint(1000000, 5000000, size)
    })
    
    stocks_data = {'600938': df}
    financial_data = {'600938': {'revenue_growth': 15, 'profit_growth': 20, 'roe': 18, 'gross_margin': 30, 'debt_ratio': 40}}
    macro_data = {'brent_crude': 104.66, 'pmi': 51.2}
    
    # 测试计算时间
    start = time.time()
    results = engine.calculate_all(stocks_data, financial_data, macro_data)
    elapsed = time.time() - start
    
    print(f"  • 数据量{size}条：{elapsed:.3f}s | {size/elapsed:.0f}条/秒")

## 4️⃣ 内存使用测试

In [ ]:
print("\n【测试 4】内存使用基准...")

# 开始内存追踪
tracemalloc.start()

# 执行完整流程
stocks_data = data_loader.load_all_stocks()
macro_data = data_loader.load_all_macro()
financial_data = data_loader.load_all_financial()

results = engine.calculate_all(stocks_data, financial_data, macro_data)

# 获取内存使用
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\n📊 内存使用：")
print(f"  • 当前内存：{current / 1024 / 1024:.2f} MB")
print(f"  • 峰值内存：{peak / 1024 / 1024:.2f} MB")

## 5️⃣ 系统吞吐量测试

In [ ]:
print("\n【测试 5】系统吞吐量...")

# 测试并发处理能力（模拟）
import concurrent.futures

def process_stock(code):
    df = data_loader.load_stock_daily(code, min_days=50)
    if df is not None and len(df) > 0:
        return len(df)
    return 0

test_codes = ['600938', '601899', '300750', '600406', '000400', '601088', '600256', '600803']

# 串行处理
start = time.time()
for code in test_codes:
    process_stock(code)
serial_time = time.time() - start

# 并行处理
start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(process_stock, test_codes))
parallel_time = time.time() - start

print(f"\n📊 吞吐量测试：")
print(f"  • 串行处理：{serial_time:.3f}s")
print(f"  • 并行处理：{parallel_time:.3f}s")
print(f"  • 加速比：{serial_time/parallel_time:.1f}x")

## 📊 测试总结

In [ ]:
print("\n" + "="*60)
print("📋 性能基准测试总结")
print("="*60)
print("✅ 缓存性能：写入/读取正常")
print("✅ 数据加载：缓存命中提升显著")
print("✅ 计算引擎：处理速度达标")
print(f"✅ 内存使用：峰值{peak / 1024 / 1024:.2f} MB")
print(f"✅ 系统吞吐：并行加速{serial_time/parallel_time:.1f}x")
print("="*60)

# 清理资源
db.close()
cache.clear()
print("\n✅ 资源已清理")
print("\n🎉 性能测试完成！")